# MMAudio Urban Soundscape Fine-tuning

Unified Google Colab workflow for the manually reviewed, non-overlapping 8-second dataset. The primary model is `small_44k`; all large artifacts remain in Google Drive.

## 1. Mount Drive and sync code

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/yuqiliang/mmaudio-finetune.git"
BRANCH = "colab"
REPO_DIR = Path("/content/MMAudio")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
%cd /content/MMAudio

## 2. Load the central configuration

In [ ]:
from fine_tune.urban_paths import (
    AUDIO_SR,
    CLIP_SEQ_LEN,
    DRIVE_ROOT,
    EVAL_DIR,
    EXT_WEIGHTS_DIR,
    FINAL_MANIFEST,
    HF_CACHE_DIR,
    LATENT_DIR,
    LATENT_SEQ_LEN,
    LOG_DIR,
    MANIFEST_DIR,
    MODEL,
    MODEL_WEIGHTS,
    MODE,
    OUTPUT_DIR,
    REPO_ROOT,
    SYNCHFORMER_WEIGHTS,
    SYNC_SEQ_LEN,
    TORCH_CACHE_DIR,
    VAE_WEIGHTS,
    VIDEO_DIR,
    ensure_drive_dirs,
    split_latent_dirs,
    split_manifests,
    split_tsvs,
)

import os

ensure_drive_dirs()
os.environ.setdefault("HF_HOME", str(HF_CACHE_DIR))
os.environ.setdefault("TORCH_HOME", str(TORCH_CACHE_DIR))
print("Drive root:", DRIVE_ROOT)
print("Dataset:", VIDEO_DIR)
print("Final manifest:", FINAL_MANIFEST)
print("Model cache:", HF_CACHE_DIR)
print("Model:", MODEL, MODE)
print("Sequence lengths:", LATENT_SEQ_LEN, CLIP_SEQ_LEN, SYNC_SEQ_LEN)

assert MODEL == "small_44k"
assert MODE == "44k"
assert (LATENT_SEQ_LEN, CLIP_SEQ_LEN, SYNC_SEQ_LEN) == (345, 64, 192)
assert AUDIO_SR == 44100

## 3. Validate the final synced manifest

The local review and QC tools must finish before this point. This check reads exact manifest paths and never scans the full Drive directory.

In [ ]:
import pandas as pd

assert FINAL_MANIFEST.is_file(), f"Missing final synced manifest: {FINAL_MANIFEST}"
manifest = pd.read_csv(FINAL_MANIFEST)
required = {"training_id", "split", "relative_path"}
assert required.issubset(manifest.columns), sorted(required - set(manifest.columns))
assert manifest["training_id"].is_unique
assert set(manifest["split"]) <= {"train", "val", "test"}
resolved_paths = manifest["relative_path"].map(lambda value: VIDEO_DIR / value)
assert resolved_paths.map(Path.is_file).all()

summary = manifest.groupby("split").agg(
    clips=("training_id", "size"),
    sources=("source_file", "nunique"),
)
summary["hours"] = summary["clips"] * 8 / 3600
display(summary)

source_split_counts = manifest.groupby("source_file")["split"].nunique()
assert source_split_counts.max() == 1, "A source video appears in multiple splits"

## 4. Download and link 44.1 kHz weights

In [ ]:
from mmaudio.utils.download_utils import download_model_if_needed

weight_links = {
    REPO_ROOT / "weights" / "mmaudio_small_44k.pth": MODEL_WEIGHTS,
    REPO_ROOT / "ext_weights" / "v1-44.pth": VAE_WEIGHTS,
    REPO_ROOT / "ext_weights" / "synchformer_state_dict.pth": SYNCHFORMER_WEIGHTS,
}

for repo_path, drive_path in weight_links.items():
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    repo_path.parent.mkdir(parents=True, exist_ok=True)
    if not repo_path.exists() and not repo_path.is_symlink():
        repo_path.symlink_to(drive_path)
    if not drive_path.exists():
        download_model_if_needed(repo_path)
    print(drive_path.name, f"{drive_path.stat().st_size / 2**20:.1f} MiB")

## 5. Check the GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "Select a GPU runtime before extraction or training"
properties = torch.cuda.get_device_properties(0)
print(torch.cuda.get_device_name(0))
print(f"GPU memory: {properties.total_memory / 2**30:.1f} GiB")

## 6. Debug feature extraction

Run four training clips first. The expected outputs are audio latents `[N, 345, 40]`, CLIP `[N, 64, 1024]`, and sync `[N, 192, 768]`.

In [ ]:
debug_dir = LATENT_DIR / "debug_train"
debug_memmap = debug_dir / "training_latents"

if debug_memmap.exists():
    print("Debug memmap already exists:", debug_memmap)
else:
    command = [
        sys.executable,
        str(REPO_ROOT / "fine_tune" / "extract_video_training_latents_colab.py"),
        "--manifest", str(split_manifests()["train"]),
        "--path_column", "relative_path",
        "--path_root", str(VIDEO_DIR),
        "--latent_dir", str(debug_dir),
        "--debug_limit", "4",
        "--batch_size", "1",
        "--num_workers", "0",
        "--mode", MODE,
        "--audio_sr", str(AUDIO_SR),
        "--vae_path", str(VAE_WEIGHTS),
        "--synchformer_ckpt", str(SYNCHFORMER_WEIGHTS),
    ]
    subprocess.run(command, cwd=REPO_ROOT, check=True)

In [ ]:
from tensordict import TensorDict

debug_td = TensorDict.load_memmap(debug_memmap)
debug_shapes = {key: list(debug_td[key].shape) for key in debug_td.keys()}
print(debug_shapes)
assert debug_shapes["mean"][1:] == [345, 40]
assert debug_shapes["std"][1:] == [345, 40]
assert debug_shapes["clip_features"][1:] == [64, 1024]
assert debug_shapes["sync_features"][1:] == [192, 768]
assert debug_shapes["text_features"][1] == 77

## 7. Extract train, validation, and test latents

Each split receives a separate memmap in exactly the same order as its TSV. Existing completed memmaps are skipped.

In [ ]:
for split in ("train", "val", "test"):
    latent_root = split_latent_dirs()[split]
    memmap = latent_root / "training_latents"
    if memmap.exists():
        print(f"{split}: skip existing {memmap}")
        continue
    command = [
        sys.executable,
        str(REPO_ROOT / "fine_tune" / "extract_video_training_latents_colab.py"),
        "--manifest", str(split_manifests()[split]),
        "--path_column", "relative_path",
        "--path_root", str(VIDEO_DIR),
        "--latent_dir", str(latent_root),
        "--batch_size", "1",
        "--num_workers", "0",
        "--mode", MODE,
        "--audio_sr", str(AUDIO_SR),
        "--vae_path", str(VAE_WEIGHTS),
        "--synchformer_ckpt", str(SYNCHFORMER_WEIGHTS),
    ]
    subprocess.run(command, cwd=REPO_ROOT, check=True)

In [ ]:
for split in ("train", "val", "test"):
    td = TensorDict.load_memmap(split_latent_dirs()[split] / "training_latents")
    tsv_rows = len(pd.read_csv(split_tsvs()[split], sep="\t"))
    shapes = {key: list(td[key].shape) for key in td.keys()}
    print(split, tsv_rows, shapes)
    assert shapes["mean"] == [tsv_rows, 345, 40]
    assert shapes["std"] == [tsv_rows, 345, 40]
    assert shapes["clip_features"] == [tsv_rows, 64, 1024]
    assert shapes["sync_features"] == [tsv_rows, 192, 768]
    assert shapes["text_features"][0] == tsv_rows

## 8. Start a fresh fine-tuning run

The default is batch size 1 for safer Colab memory use. Override it with `BATCH_SIZE=2` only after the debug run fits comfortably.

In [ ]:
environment = {
    **dict(__import__("os").environ),
    "EXP_ID": "urban_soundscape_small44k_nonstride_v1",
    "BATCH_SIZE": "1",
    "LEARNING_RATE": "2e-5",
    "NUM_ITERATIONS": "18000",
}
subprocess.run(
    ["bash", str(REPO_ROOT / "scripts" / "run_colab_finetune.sh")],
    cwd=REPO_ROOT,
    env=environment,
    check=True,
)

## 9. Resume an interrupted run

Use a full optimizer checkpoint (`*_ckpt_*.pth`), not an EMA-only weights file.

In [ ]:
CHECKPOINT_PATH = None  # Example: OUTPUT_DIR / "experiment" / "experiment_ckpt_last.pth"

if CHECKPOINT_PATH is None:
    print("Set CHECKPOINT_PATH before running this cell.")
else:
    resume_environment = {
        **dict(__import__("os").environ),
        "CHECKPOINT_PATH": str(CHECKPOINT_PATH),
        "EXP_ID": "urban_soundscape_small44k_nonstride_v1_continue",
    }
    subprocess.run(
        ["bash", str(REPO_ROOT / "scripts" / "run_colab_continue.sh")],
        cwd=REPO_ROOT,
        env=resume_environment,
        check=True,
    )

## 10. Evaluate a checkpoint

In [ ]:
EXP_ID = "urban_soundscape_small44k_nonstride_v1"
CHECKPOINT = OUTPUT_DIR / EXP_ID / f"{EXP_ID}_ema_final.pth"
EVALUATION_OUTPUT = EVAL_DIR / EXP_ID
EVALUATION_OUTPUT.mkdir(parents=True, exist_ok=True)

test_rows = pd.read_csv(split_manifests()["test"])
assert CHECKPOINT.is_file(), CHECKPOINT
for relative_path in test_rows["relative_path"].head(5):
    video_path = VIDEO_DIR / relative_path
    subprocess.run(
        [
            sys.executable,
            str(REPO_ROOT / "demo_custom_ckpt.py"),
            "--variant", MODEL,
            "--weights", str(CHECKPOINT),
            "--video", str(video_path),
            "--duration", "8",
            "--output", str(EVALUATION_OUTPUT),
            "--seed", "42",
            "--skip_video_composite",
        ],
        cwd=REPO_ROOT,
        check=True,
    )
print("Evaluation outputs:", EVALUATION_OUTPUT)